In [2]:
import pandas as pd
import glob
import os
import requests
import rasterio
import numpy as np
import json

In [3]:
import pandas as pd
import unicodedata

# 1. Funcție pentru eliminarea diacriticelor și curățarea textului
def normalizeaza_nume(text):
    if not isinstance(text, str): return text
    # Elimina spatii albe de la inceput/sfarsit
    text = text.strip()
    # Normalizeaza caracterele (ex: ș -> s, ț -> t)
    text = ''.join(c for c in unicodedata.normalize('NFKD', text)
                  if unicodedata.category(c) != 'Mn')
    return text

# 2. Încarcă datele
df_copaci = pd.read_csv('date_meri.csv')
df_sol = pd.read_csv('sol_judete_romania.csv')
df_clima = pd.read_csv('clima_nasa_judete_romania.csv')

# 3. Aplică normalizarea pe toate cele 3 tabele
df_copaci['Judet'] = df_copaci['Judet'].apply(normalizeaza_nume)
df_sol['Judet'] = df_sol['Judet'].apply(normalizeaza_nume)
df_clima['Judet'] = df_clima['Judet'].apply(normalizeaza_nume)

# 4. Asigură-te că coloana 'An' este de același tip (Integer)
df_copaci['An'] = df_copaci['An'].astype(int)
df_clima['An'] = df_clima['An'].astype(int)

# 5. Realizează Merge-ul pas cu pas
# Prima data Copaci + Sol (pe Judet)
df_merge_1 = pd.merge(df_copaci, df_sol, on='Judet', how='inner')
print(f"Dupa primul merge (Sol): {len(df_merge_1)} randuri")

# Apoi rezultatul + Clima (pe Judet si An)
df_final = pd.merge(df_merge_1, df_clima, on=['Judet', 'An'], how='inner')
print(f"Dupa al doilea merge (Clima): {len(df_final)} randuri")

# 6. Salvare
if len(df_final) > 0:
    df_final.to_csv("dataset_full.csv", index=False)
    print("Succes! Fișierul dataset_full.csv conține acum datele unite.")
else:
    print("EROARE: Tot nu avem potriviri. Verifică dacă anii din Clima se suprapun cu anii din Copaci.")

Dupa primul merge (Sol): 1025 randuri
Dupa al doilea merge (Clima): 1025 randuri
Succes! Fișierul dataset_full.csv conține acum datele unite.


In [5]:

import importlib
import config

# Reîncărcăm fișierul de configurare pentru a prelua ultimele modificări
importlib.reload(config)
from config import PARAMETRI_COPACI, PARAMETRI_SOL, PARAMETRI_CLIMA

# 1. Extragem cheile din dicționare (păstrează ordinea din config.py)
cols_copaci = list(PARAMETRI_COPACI.keys())
cols_sol = list(PARAMETRI_SOL.keys())

# 2. Aplatizăm dicționarul de climă
cols_clima = []
for anotimp_dict in PARAMETRI_CLIMA.values():
    cols_clima.extend(list(anotimp_dict.keys()))

# 3. CONSTRUIREA LISTEI FINALE ÎN ORDINEA DORITĂ
# Punem explicit 'Judet' și 'An' la început
toate_coloanele_ordonate = ['Judet', 'An'] + cols_copaci + cols_sol + cols_clima

# 4. Curățăm lista de eventuale duplicate păstrând ordinea
# (Ex: dacă 'An' apare și în lista de climă, îl lăsăm doar la început)
coloane_finale = []
for col in toate_coloanele_ordonate:
    if col not in coloane_finale:
        coloane_finale.append(col)

# 5. Aplicăm filtrarea pe df_final
if 'df_final' in locals() or 'df_final' in globals():
    # Păstrăm doar coloanele care chiar există în tabel
    coloane_de_extras = [c for c in coloane_finale if c in df_final.columns]
    
    df_redus = df_final[coloane_de_extras]

    # Salvare în CSV
    df_redus.to_csv('date_predictie_meri.csv', index=False)
    
    print("✅ Fișier salvat cu ordinea coloanelor:")
    print(f"1. Identificare: Judet, An")
    print(f"2. Copaci: {cols_copaci}")
    print(f"3. Sol: {cols_sol}")
    print(f"4. Clima: {len(cols_clima)} coloane")
else:
    print("❌ Eroare: df_final nu a fost găsit în memorie.")

✅ Fișier salvat cu ordinea coloanelor:
1. Identificare: Judet, An
2. Copaci: ['Numar_Pomi', 'Kg_pe_Pom', 'Tone_Totale', 'Rootstock_ID']
3. Sol: ['phh2o', 'clay', 'sand', 'silt', 'soc', 'nitrogen', 'cec']
4. Clima: 18 coloane
